# LeetCode 3842 — Toggle Light Bulbs bằng Python

Link đề: https://leetcode.com/problems/toggle-light-bulbs/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Simulation, Parity

Skeleton LeetCode:

```python
class Solution:
    def toggleLightBulbs(self, bulbs: List[int]) -> List[int]:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có 100 bóng đèn đánh số `1..100`, ban đầu **tắt hết**; mỗi phần tử của `bulbs` là một lần **bấm công tắc** (bật nếu đang tắt, tắt nếu đang bật). Kết quả là danh sách bóng **đang sáng** theo thứ tự tăng dần.
2. Nhận ra tính chất **chẵn/lẻ (parity)**: một bóng sáng khi và chỉ khi nó được bấm một số **lẻ** lần.
3. Biết dùng `list` boolean, `Counter`, `set` với phép `^` (hiệu đối xứng), bitmask `XOR` và `groupby` để biểu diễn trạng thái.
4. Viết 5 cách khác nhau và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: quên sắp xếp, nhầm "từng được bấm" với "đang sáng", đếm `== 1` thay vì lẻ, nhầm chỉ số 0/1.

## 2. Đề bài & ví dụ

Có 100 bóng đèn đánh số từ `1` đến `100`, ban đầu **đều tắt**. Cho mảng `bulbs`. Với mỗi phần tử `bulbs[i]` theo thứ tự từ trái sang phải: nếu bóng đèn đó đang **tắt** thì **bật** lên; nếu đang **bật** thì **tắt** đi. Trả về danh sách số hiệu các bóng đèn **đang bật** sau khi xử lý hết, **sắp xếp tăng dần**. Nếu không còn bóng nào sáng thì trả về danh sách rỗng.

| Ví dụ | bulbs | Output | Giải thích |
|---|---|---|---|
| 1 | `[10,30,20,10]` | `[20,30]` | Bóng 10, 30, 20 được bật; lần bấm thứ hai vào 10 tắt nó đi; còn 20 và 30 |
| 2 | `[100,100]` | `[]` | Bóng 100 bật rồi tắt, không bóng nào sáng |

Constraints: `1 <= bulbs.length <= 100`; `1 <= bulbs[i] <= 100`.

In [1]:
state = {}
for b in [10, 30, 20, 10]:
    state[b] = not state.get(b, False)
    print(f"bấm {b:3d} -> {'sáng' if state[b] else 'tắt'}")
print(sorted(k for k, v in state.items() if v))

bấm  10 -> sáng
bấm  30 -> sáng
bấm  20 -> sáng
bấm  10 -> tắt
[20, 30]


## 3. Kiến thức nền

### 3.1. Danh sách boolean làm "bảng trạng thái"

Một `list` gồm `True/False`, vị trí `b` lưu trạng thái của bóng `b`. Vì bóng đánh số từ `1`, ta tạo `101` phần tử (bỏ vị trí `0`) để khỏi trừ 1. Phép đảo trạng thái: `lamp[b] = not lamp[b]`.

In [2]:
lamp = [False] * 101
lamp[10] = not lamp[10]
lamp[10] = not lamp[10]
lamp[20] = not lamp[20]
print([i for i in range(101) if lamp[i]])

[20]


### 3.2. `Counter` đếm số lần xuất hiện

`collections.Counter(list)` trả về từ điển `{phần tử: số lần}`. Dùng để biết mỗi bóng được bấm bao nhiêu lần.

In [3]:
from collections import Counter
c = Counter([10, 30, 20, 10])
print(c, c[10], c[99])      # phần tử không có -> 0
print([k for k, v in c.items() if v % 2 == 1])

Counter({10: 2, 30: 1, 20: 1}) 2 0
[30, 20]


### 3.3. `set` và phép hiệu đối xứng `^`

`A ^ B` giữ các phần tử thuộc đúng một trong hai tập. Với `A = {10, 30}` và `{10}`: kết quả `{30}` (10 bị "xóa"). Phép `s ^= {b}` nghĩa là "nếu `b` đã có trong `s` thì xóa, chưa có thì thêm" — đúng ý nghĩa **bật/tắt**.

In [4]:
s = set()
for b in [10, 30, 20, 10]:
    s ^= {b}
    print(f"sau {b}: {sorted(s)}")

sau 10: [10]
sau 30: [10, 30]
sau 20: [10, 20, 30]
sau 10: [20, 30]


### 3.4. Bitmask và `XOR`

Dùng một số nguyên lớn, bit thứ `b` là trạng thái bóng `b`. `mask ^= 1 << b` đảo đúng bit `b`. Đọc kết quả: `(mask >> b) & 1`.

In [5]:
mask = 0
for b in [10, 30, 20, 10]:
    mask ^= 1 << b
print(bin(mask))
print([b for b in range(1, 101) if (mask >> b) & 1])

0b1000000000100000000000000000000
[20, 30]


### 3.5. `sorted` và `itertools.groupby`

`sorted(iterable)` trả list mới đã sắp xếp (không đổi list gốc). `groupby` gom các phần tử **liên tiếp bằng nhau**, nên phải `sorted` trước.

In [6]:
from itertools import groupby
arr = sorted([10, 30, 20, 10])
print(arr, [(k, len(list(g))) for k, g in groupby(arr)])

[10, 10, 20, 30] [(10, 2), (20, 1), (30, 1)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Bấm một bóng **hai lần** thì nó trở về trạng thái ban đầu (tắt). Bấm **ba lần** thì như bấm một lần. Vậy trạng thái cuối cùng chỉ phụ thuộc vào **số lần bấm là chẵn hay lẻ**, không phụ thuộc thứ tự bấm hay các bóng khác (các bóng độc lập với nhau).

Tính tay `bulbs = [10, 30, 20, 10]`:

| bóng | số lần bấm | chẵn/lẻ | trạng thái cuối |
|---|---|---|---|
| 10 | 2 | chẵn | tắt |
| 20 | 1 | lẻ | sáng |
| 30 | 1 | lẻ | sáng |

Kết quả `[20, 30]`. Ví dụ khác `[7, 7, 7, 2]`: bóng 7 bấm 3 lần (lẻ, sáng), bóng 2 bấm 1 lần (sáng) -> `[2, 7]`.

In [7]:
from collections import Counter
for bulbs in ([10, 30, 20, 10], [7, 7, 7, 2], [100, 100]):
    c = Counter(bulbs)
    print(bulbs, dict(c), "->", sorted(b for b, k in c.items() if k % 2 == 1))

[10, 30, 20, 10] {10: 2, 30: 1, 20: 1} -> [20, 30]
[7, 7, 7, 2] {7: 3, 2: 1} -> [2, 7]
[100, 100] {100: 2} -> []


### Một cách nhìn khác: XOR là cộng mod 2

Gán `1` cho "sáng", `0` cho "tắt". Mỗi lần bấm là cộng thêm `1` rồi lấy dư mod 2: `0 -> 1`, `1 -> 0`. Phép cộng mod 2 chính là `XOR`. Vì `XOR` có tính giao hoán và kết hợp nên **thứ tự bấm không quan trọng** — ta có thể xáo trộn `bulbs` thoải mái mà kết quả không đổi (sẽ kiểm tra ở mục 12).

In [8]:
for a in (0, 1):
    for b in (0, 1):
        print(f"{a} XOR {b} = {a ^ b}    ({a} + {b}) % 2 = {(a + b) % 2}")

0 XOR 0 = 0    (0 + 0) % 2 = 0
0 XOR 1 = 1    (0 + 1) % 2 = 1
1 XOR 0 = 1    (1 + 0) % 2 = 1
1 XOR 1 = 0    (1 + 1) % 2 = 0


## 5. Cách 1 — Brute Force / baseline

Mô phỏng đúng như đề: một mảng `lamp` gồm 101 giá trị `False`. Với mỗi lần bấm đảo `lamp[b]`; cuối cùng quét `b = 1..100` và lấy các bóng `True` (đã tự động theo thứ tự tăng).

Nút thắt: không có nút thắt đáng kể ở đây (`O(n + 100)`); điều ta muốn cải thiện là **độ gọn** (không phụ thuộc cố định 100 bóng) và hiểu bản chất "chẵn/lẻ".

In [9]:
class SolutionBrute:
    def toggleLightBulbs(self, bulbs):
        lamp = [False] * 101               # lamp[b]: bóng b đang sáng?
        for b in bulbs:
            lamp[b] = not lamp[b]
        return [b for b in range(1, 101) if lamp[b]]

print(SolutionBrute().toggleLightBulbs([10, 30, 20, 10]), SolutionBrute().toggleLightBulbs([100, 100]))

[20, 30] []


**Độ phức tạp:** Time `O(n + 100)` (`n = len(bulbs)`), Space `O(100)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** trạng thái cuối của bóng `b` chỉ phụ thuộc `số lần bấm b` mod 2. Do đó:

- Có thể **đếm** số lần bấm rồi lọc lẻ (`Counter`),
- hoặc giữ một **tập các bóng đang sáng** và dùng `^` để thêm/bớt,
- hoặc dùng **bitmask** với `XOR`,
- hoặc **sắp xếp** rồi gom nhóm các phần tử bằng nhau.

Invariant: sau khi xử lý `i` lần bấm đầu tiên, tập `on` chứa đúng các bóng được bấm một số lẻ lần trong `bulbs[:i]`. Bấm thêm bóng `b`: số lần bấm `b` tăng 1 nên tính lẻ/chẵn của nó đổi -> thêm vào `on` nếu chưa có, bỏ ra nếu đã có.

## 7. Các cách tiếp theo

### Cách 2 — `Counter` và lọc số lần lẻ

Đếm số lần xuất hiện, giữ bóng có `count % 2 == 1`, rồi `sorted`.

In [10]:
from collections import Counter

class SolutionCounter:
    def toggleLightBulbs(self, bulbs):
        cnt = Counter(bulbs)
        return sorted(b for b, k in cnt.items() if k % 2 == 1)

print(SolutionCounter().toggleLightBulbs([10, 30, 20, 10]), SolutionCounter().toggleLightBulbs([100, 100]))

[20, 30] []


**Độ phức tạp:** Time `O(n + m log m)` với `m` là số bóng khác nhau (`m <= 100`), Space `O(m)`.

### Cách 3 — Tập "đang sáng" với `^=`

Duy trì `on = set()`; mỗi lần bấm `on ^= {b}`. Cuối cùng `sorted(on)`.

In [11]:
class SolutionSet:
    def toggleLightBulbs(self, bulbs):
        on = set()
        for b in bulbs:
            on ^= {b}            # có thì bỏ, chưa có thì thêm
        return sorted(on)

print(SolutionSet().toggleLightBulbs([10, 30, 20, 10]), SolutionSet().toggleLightBulbs([100, 100]))

[20, 30] []


**Độ phức tạp:** Time `O(n + m log m)`, Space `O(m)`.

### Cách 4 — Bitmask `XOR`

Một số nguyên `mask`; `mask ^= 1 << b`. Đọc kết quả bằng cách duyệt các bit `1..100`. Cách này ngắn gọn khi các giá trị nhỏ và cho thấy bản chất "XOR = cộng modulo 2".

In [12]:
class SolutionMask:
    def toggleLightBulbs(self, bulbs):
        mask = 0
        for b in bulbs:
            mask ^= 1 << b
        return [b for b in range(1, 101) if (mask >> b) & 1]

print(SolutionMask().toggleLightBulbs([10, 30, 20, 10]), SolutionMask().toggleLightBulbs([100, 100]))

[20, 30] []


**Độ phức tạp:** Time `O(n + 100)`, Space `O(1)` số nguyên 101 bit (hằng số).

### Cách 5 — Sắp xếp rồi gom nhóm (`groupby`)

Sắp xếp `bulbs`, các lần bấm cùng một bóng đứng cạnh nhau. Với mỗi nhóm có độ dài lẻ, bóng đó sáng. Kết quả tự động đã tăng dần.

In [13]:
from itertools import groupby

class SolutionGroup:
    def toggleLightBulbs(self, bulbs):
        res = []
        for b, grp in groupby(sorted(bulbs)):
            if len(list(grp)) % 2 == 1:
                res.append(b)
        return res

print(SolutionGroup().toggleLightBulbs([10, 30, 20, 10]), SolutionGroup().toggleLightBulbs([100, 100]))

[20, 30] []


**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 với `bulbs = [10, 30, 20, 10]`:

| Bước | bấm | `on` trước | hành động | `on` sau |
|---|---|---|---|---|
| 1 | 10 | `{}` | thêm 10 | `{10}` |
| 2 | 30 | `{10}` | thêm 30 | `{10, 30}` |
| 3 | 20 | `{10, 30}` | thêm 20 | `{10, 20, 30}` |
| 4 | 10 | `{10, 20, 30}` | bỏ 10 | `{20, 30}` |

In [14]:
def trace_toggle(bulbs):
    on = set()
    for step, b in enumerate(bulbs, 1):
        before = sorted(on)
        action = "tắt" if b in on else "bật"
        on ^= {b}
        print(f"bước {step}: bấm {b:3d} ({action}) {before} -> {sorted(on)}")
    return sorted(on)

assert trace_toggle([10, 30, 20, 10]) == [20, 30]
assert trace_toggle([100, 100]) == []

bước 1: bấm  10 (bật) [] -> [10]
bước 2: bấm  30 (bật) [10] -> [10, 30]
bước 3: bấm  20 (bật) [10, 30] -> [10, 20, 30]
bước 4: bấm  10 (tắt) [10, 20, 30] -> [20, 30]
bước 1: bấm 100 (bật) [] -> [100]
bước 2: bấm 100 (tắt) [100] -> []


In [15]:
def trace_mask(bulbs):
    mask = 0
    for b in bulbs:
        mask ^= 1 << b
        print(f"bấm {b:3d}: các bit bật = {[i for i in range(1, 101) if (mask >> i) & 1]}")
    return mask

trace_mask([5, 6, 5, 7, 7, 7])

bấm   5: các bit bật = [5]
bấm   6: các bit bật = [5, 6]
bấm   5: các bit bật = [6]
bấm   7: các bit bật = [6, 7]
bấm   7: các bit bật = [6]
bấm   7: các bit bật = [6, 7]


192

## 9. Code nộp LeetCode

Dùng Cách 2 (`Counter` + lọc lẻ + `sorted`): ngắn và đúng bản chất chẵn/lẻ.

In [16]:
from typing import List, Optional
from collections import Counter


class Solution:
    def toggleLightBulbs(self, bulbs: List[int]) -> List[int]:
        cnt = Counter(bulbs)
        return sorted(b for b, k in cnt.items() if k % 2 == 1)


print(Solution().toggleLightBulbs([10, 30, 20, 10]), Solution().toggleLightBulbs([100, 100]))

[20, 30] []


## 10. Vì sao thuật toán đúng

1. **Các bóng độc lập.** Bấm bóng `b` không ảnh hưởng bóng khác, nên trạng thái cuối của `b` chỉ phụ thuộc dãy các lần bấm **chính nó**.
2. **Bấm hai lần = không đổi.** Bóng tắt -> bật -> tắt; bóng bật -> tắt -> bật. Nên mỗi cặp bấm triệt tiêu nhau.
3. **Hệ quả:** sau `k` lần bấm, bóng sáng nếu `k` lẻ (ban đầu tắt), tắt nếu `k` chẵn. `Counter` cho đúng `k`, nên lọc `k % 2 == 1` đúng.
4. **Thứ tự đầu ra.** `sorted` đảm bảo tăng dần như đề yêu cầu; danh sách rỗng khi không có bóng lẻ.
5. Các cách còn lại (`^=`, `XOR`, `groupby`, mảng boolean) đều là cách khác nhau để tính cùng một chẵn/lẻ.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(bulbs)` và `m` là số bóng khác nhau xuất hiện (`m <= min(n, 100)`).

- Cách 1: `O(n + 100)` thời gian, `O(100)` bộ nhớ.
- Cách 2, 3: `O(n + m log m)` thời gian (đếm/duyệt + sắp xếp kết quả), `O(m)` bộ nhớ.
- Cách 4: `O(n + 100)` thời gian, bộ nhớ hằng số.
- Cách 5: `O(n log n)` thời gian vì sắp xếp mảng đầu vào, `O(n)` bộ nhớ.

Với `n <= 100` mọi cách đều tức thời. Không có khác biệt best / worst lớn; chỉ Cách 5 chịu chi phí `n log n`.

## 12. Bộ test

Đối chiếu 5 cách + code nộp với ví dụ đề, biên và 300 input ngẫu nhiên đúng constraints (`1 <= len <= 100`, giá trị `1..100`).

In [17]:
import random

solutions = [SolutionBrute(), SolutionCounter(), SolutionSet(), SolutionMask(), SolutionGroup(), Solution()]

fixed = [
    ([10, 30, 20, 10], [20, 30]),
    ([100, 100], []),
    ([1], [1]),
    ([100], [100]),
    ([1, 1, 1], [1]),                 # bấm 3 lần: sáng
    ([5, 5, 5, 5], []),               # bấm 4 lần: tắt
    ([3, 2, 1], [1, 2, 3]),           # phải sắp xếp tăng dần
    ([7, 7, 7, 2], [2, 7]),
    (list(range(1, 101)), list(range(1, 101))),
    (list(range(1, 101)) * 1, list(range(1, 101))),
]
for bulbs, expected in fixed:
    for s in solutions:
        assert s.toggleLightBulbs(list(bulbs)) == expected, (type(s).__name__, bulbs)

random.seed(38420)
for _ in range(300):
    n = random.randint(1, 100)
    hi = random.choice([3, 10, 100])
    bulbs = [random.randint(1, hi) for _ in range(n)]
    outs = [s.toggleLightBulbs(list(bulbs)) for s in solutions]
    assert all(o == outs[0] for o in outs)
    assert outs[0] == sorted(outs[0])
print("tất cả test OK")

tất cả test OK


### Test thuộc tính

(1) Xáo trộn thứ tự `bulbs` không đổi kết quả. (2) Thêm **hai lần liên tiếp** cùng một bóng không đổi kết quả. (3) Hai bấm hai bóng khác nhau cho kết quả giống hợp của chúng khi chưa trùng.

In [18]:
random.seed(3842)
for _ in range(300):
    n = random.randint(1, 50)
    bulbs = [random.randint(1, 100) for _ in range(n)]
    base = Solution().toggleLightBulbs(bulbs)
    shuffled = bulbs[:]
    random.shuffle(shuffled)
    assert Solution().toggleLightBulbs(shuffled) == base
    x = random.randint(1, 100)
    assert Solution().toggleLightBulbs(bulbs + [x, x]) == base
    assert Solution().toggleLightBulbs(bulbs + [x]) != base
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên sắp xếp kết quả

Nếu duyệt `Counter` theo thứ tự xuất hiện đầu tiên thì kết quả không tăng dần. Với `[10, 30, 20, 10]` ta nhận `[30, 20]` thay vì `[20, 30]`.

In [19]:
def wrong_unsorted(bulbs):
    cnt = Counter(bulbs)
    return [b for b, k in cnt.items() if k % 2 == 1]         # SAI: thiếu sorted

print(wrong_unsorted([10, 30, 20, 10]), "!=", Solution().toggleLightBulbs([10, 30, 20, 10]))
assert wrong_unsorted([10, 30, 20, 10]) == [30, 20] and Solution().toggleLightBulbs([10, 30, 20, 10]) == [20, 30]

[30, 20] != [20, 30]


### Lỗi 2: coi "từng được bấm" là "đang sáng"

`sorted(set(bulbs))` trả về mọi bóng đã bấm, kể cả những bóng đã bị tắt lại.

In [20]:
def wrong_distinct(bulbs):
    return sorted(set(bulbs))                                 # SAI: bỏ qua việc tắt lại

print(wrong_distinct([10, 30, 20, 10]), "!=", Solution().toggleLightBulbs([10, 30, 20, 10]))
assert wrong_distinct([100, 100]) == [100] and Solution().toggleLightBulbs([100, 100]) == []

[10, 20, 30] != [20, 30]


### Lỗi 3: lọc `count == 1` thay vì `count` lẻ

Bóng bấm 3 lần vẫn sáng. Điều kiện `== 1` loại nhầm nó.

In [21]:
def wrong_eq_one(bulbs):
    cnt = Counter(bulbs)
    return sorted(b for b, k in cnt.items() if k == 1)        # SAI: phải k % 2 == 1

print(wrong_eq_one([1, 1, 1, 2]), "!=", Solution().toggleLightBulbs([1, 1, 1, 2]))
assert wrong_eq_one([1, 1, 1, 2]) == [2] and Solution().toggleLightBulbs([1, 1, 1, 2]) == [1, 2]

[2] != [1, 2]


### Lỗi 4: lệch chỉ số khi dùng mảng boolean

Mảng chỉ có 100 phần tử thì `lamp[100]` ngoài biên (bóng đánh số từ `1` đến `100`). Cần 101 phần tử hoặc trừ 1 khi truy cập.

In [22]:
def wrong_size(bulbs):
    lamp = [False] * 100                                      # SAI: thiếu chỗ cho bóng 100
    for b in bulbs:
        lamp[b] = not lamp[b]
    return [b for b in range(1, 101) if lamp[b]]

try:
    wrong_size([100])
except IndexError as e:
    print("IndexError:", e)
assert Solution().toggleLightBulbs([100]) == [100]

IndexError: list index out of range


### Lỗi 5: dùng list `append/remove` và trả về theo thứ tự bấm

Giữ list `on` rồi `append`/`remove` cho đúng giá trị nhưng **thứ tự** là thứ tự bấm, không tăng dần. Ngoài ra `remove` và `in` trên list đều tốn `O(n)`.

In [23]:
def wrong_remove_while_iterating(bulbs):
    on = []
    for b in bulbs:
        if b in on:
            on.remove(b)
        else:
            on.append(b)
    return on                                                  # SAI: thiếu sorted

print(wrong_remove_while_iterating([20, 10, 5]), "!=", Solution().toggleLightBulbs([20, 10, 5]))
assert wrong_remove_while_iterating([30, 10, 30, 20]) == [10, 20]       # đúng khi vô tình đã tăng dần
assert wrong_remove_while_iterating([20, 10, 5]) == [20, 10, 5] != Solution().toggleLightBulbs([20, 10, 5])

[20, 10, 5] != [5, 10, 20]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Bật/tắt", "đảo trạng thái", "hai lần thì trở về" -> **parity** (chẵn/lẻ) và XOR.
- Hầu hết bài "toggle" có thể rút gọn thành "đếm số lần, giữ những phần tử lẻ".
- `s ^= {x}` trên `set` là mẹo ngắn gọn để thêm/bớt phần tử.
- Với giá trị nhỏ (`<= 100`), bitmask hoặc mảng boolean là lựa chọn hiệu quả.
- Luôn đọc kỹ yêu cầu về **thứ tự đầu ra** (ở đây là tăng dần); thiếu `sorted` là lỗi hay gặp.

## 15. Biến thể của bài

1. **Bulb Switcher** (319): bấm theo bội số, đáp án là số chính phương.
2. **Bulb Switcher II** (672): các thao tác bật/tắt theo luật.
3. **Single Number** (136): XOR để tìm phần tử xuất hiện lẻ lần.
4. **Find the Difference** / **Missing Number**: XOR/đếm chẵn lẻ.
5. **Remove Duplicates / Delete Pairs**: loại các cặp giống nhau bằng stack hoặc set.
6. Biến thể: `bulbs[i]` có thể lên tới `10^9` -> dùng `Counter` thay vì bitmask/mảng boolean.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mảng boolean | mô phỏng trực tiếp | `O(n + 100)` | `O(100)` | Dễ hiểu nhất |
| 2. `Counter` | đếm, lọc lẻ, sắp xếp | `O(n + m log m)` | `O(m)` | Nộp bài, giá trị lớn cũng được |
| 3. `set ^=` | thêm/bớt từng lần | `O(n + m log m)` | `O(m)` | Ngắn, đúng nghĩa bật/tắt |
| 4. Bitmask XOR | bit là trạng thái | `O(n + 100)` | `O(1)` | Giá trị nhỏ, thích bit |
| 5. Sort + groupby | gom nhóm bằng nhau | `O(n log n)` | `O(n)` | Khi mảng đã sắp xếp sẵn |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().toggleLightBulbs([4, 4, 4, 9, 9])` ra gì?

**Bài 2 (tính tay):** với `bulbs = [2, 3, 2, 3, 5, 5, 7]` bóng nào sáng?

**Bài 3 (điền TODO):** hoàn thành `count_on(bulbs)` trả về **số lượng** bóng đang sáng (không cần danh sách).

**Bài 4 (tự viết lại):** viết `toggle_with_initial_on(bulbs, initially_on)` với danh sách `initially_on` các bóng sáng sẵn lúc đầu (các bóng còn lại tắt).

**Bài 5 (mini challenge):** viết `first_time_all_off(bulbs)` trả về chỉ số `i` nhỏ nhất (đếm từ 1) sao cho sau `i` lần bấm **không còn bóng nào sáng**, hoặc `-1` nếu không bao giờ.

In [24]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().toggleLightBulbs([4, 4, 4, 9, 9]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_on_todo(bulbs):
    on = set()
    # TODO: dùng on ^= {b} rồi trả về số phần tử
    return 0

[4]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: đếm số lần bấm của `4` và của `9`. Bài 2: bóng `2` và `3` đều 2 lần.

**Mức 2 (vừa):** Bài 3: `len(on)`. Bài 4: khởi tạo `on = set(initially_on)` thay vì rỗng. Bài 5: sau mỗi lần bấm kiểm tra `not on` (tập rỗng).

**Mức 3 (gần đáp án):** Bài 1: `[4]`. Bài 2: `[7]`. Bài 4: với `bulbs = [1, 2]` và `initially_on = [1]` kết quả `[2]`. Bài 5: với `[3, 3]` kết quả `2`; với `[3]` kết quả `-1`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
assert Solution().toggleLightBulbs([4, 4, 4, 9, 9]) == [4]
assert Solution().toggleLightBulbs([2, 3, 2, 3, 5, 5, 7]) == [7]

# Bài 3
def count_on(bulbs):
    on = set()
    for b in bulbs:
        on ^= {b}
    return len(on)
assert count_on([10, 30, 20, 10]) == 2 and count_on([100, 100]) == 0

# Bài 4
def toggle_with_initial_on(bulbs, initially_on):
    on = set(initially_on)
    for b in bulbs:
        on ^= {b}
    return sorted(on)
assert toggle_with_initial_on([1, 2], [1]) == [2]
assert toggle_with_initial_on([5, 5], [5, 6]) == [5, 6]

# Bài 5
def first_time_all_off(bulbs):
    on = set()
    for i, b in enumerate(bulbs, 1):
        on ^= {b}
        if not on:
            return i
    return -1
assert first_time_all_off([3, 3]) == 2 and first_time_all_off([3]) == -1
assert first_time_all_off([1, 2, 2, 1]) == 4 and first_time_all_off([1, 1, 2]) == 2
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "bật/tắt = đếm chẵn lẻ":**

```python
from collections import Counter
cnt = Counter(bulbs)
return sorted(x for x, k in cnt.items() if k % 2 == 1)
# hoặc: on = set(); for x in bulbs: on ^= {x}; return sorted(on)
```

**Checklist trước khi submit:**

- [ ] Ban đầu mọi bóng **tắt** (nên sáng khi số lần bấm **lẻ**)?
- [ ] Lọc `k % 2 == 1`, không phải `k == 1` hay `k >= 1`?
- [ ] Kết quả đã `sorted` tăng dần, rỗng thì trả `[]`?
- [ ] Nếu dùng mảng boolean: kích thước 101?
- [ ] Đã thử `[100, 100]`, `[1, 1, 1]`, `[3, 2, 1]`?
- [ ] Chữ ký đúng `toggleLightBulbs(self, bulbs: List[int]) -> List[int]`?